# Recommendation Systems — Interview Q&A

> **Target roles:** Research Scientist · ML Engineer · RecSys Team · Foundational Modeling  
> Covers: Collaborative Filtering, Content-Based, Hybrid, Matrix Factorization, Deep RecSys, Two-Tower models, evaluation, and production.  
> ⚠️ marks trick questions.

## 1. RecSys Fundamentals

**Q1. What are the main paradigms of recommendation systems?**  
**Collaborative Filtering (CF):** Uses user-item interaction history. "Users like you also liked..." Doesn't need item features. Suffers from cold-start.  
**Content-Based Filtering:** Uses item features (genre, author, tags) to recommend similar items. Works for new items; misses serendipity; filter bubble problem.  
**Hybrid:** Combines CF and content-based. Most production systems are hybrid. Can be weighted, switching, cascade, or feature-augmented.  
**Knowledge-Based:** Uses explicit user requirements/constraints (e.g., "show me red dresses under $50"). Useful for infrequent purchases (cars, real estate).

**Q2. What is the cold-start problem and how do you address it?**  
**User cold-start:** New users with no history. Solutions: onboarding questionnaire, popular items fallback, content-based from user profile/demographics.  
**Item cold-start:** New items with no interactions. Solutions: content-based features, metadata embeddings, exploration policies (ε-greedy, Thompson sampling).  
**System cold-start:** Completely new system with no data. Solutions: knowledge-based rules, manual curation, cross-domain transfer.

**Q3. Explain user-based vs item-based collaborative filtering.**  
**User-based CF:** Find users similar to target user (cosine/Pearson similarity on rating vectors); recommend items they liked that target hasn't seen. Expensive: O(U²) comparisons. Users change more frequently than items.  
**Item-based CF:** Find items similar to what the target user liked; recommend similar items. More stable (items don't change behavior), easier to precompute, scales better. Amazon's original algorithm was item-based CF.

**Q4. What is the Pearson correlation vs cosine similarity for CF? Which is better?**  
**Cosine:** cos(u,v) = (u·v)/(‖u‖·‖v‖). Treats missing ratings as 0 — can be misleading if most users rate few items.  
**Pearson:** Correlation after subtracting each user's mean rating. Adjusts for rating bias (some users always rate high/low). Generally preferred for explicit rating data.  
**Adjusted cosine:** Subtracts item mean — useful for item-based CF.  
For implicit feedback (clicks, views), cosine on binary or frequency-weighted vectors is more common.


## 2. Matrix Factorization

**Q5. Explain matrix factorization for collaborative filtering.**  
Decompose the sparse user-item rating matrix R (U×I) into low-rank factors: R ≈ P·Qᵀ where P (U×k) are user latent factors and Q (I×k) are item latent factors. k is the latent dimension (typically 16-256). Rating prediction: r̂ᵢⱼ = pᵢ·qⱼᵀ. Learn P and Q by minimizing regularized squared error: min Σ(rᵢⱼ - pᵢqⱼᵀ)² + λ(‖P‖² + ‖Q‖²).

**Q6. How is matrix factorization trained? SVD vs SGD vs ALS.**  
**Truncated SVD:** Only works on *dense* matrices. Can fill missing values with zeros/means first (biased).  
**SGD:** Sample observed (user, item) pairs, compute gradient, update pᵢ and qⱼ. Simple, parallelizable, handles sparse data naturally.  
**ALS (Alternating Least Squares):** Fix Q, solve for P in closed form (each user independently); fix P, solve for Q. Highly parallelizable — each user/item row is independent. Better convergence for implicit feedback (iALS). Used in Spark MLlib.

**Q7. What is the SVD++ model?**  
Extends basic MF by incorporating *implicit feedback* (items rated, not just how). User factor: p̃ᵢ = pᵢ + |R(i)|^(-1/2) Σⱼ∈R(i) yⱼ where yⱼ are implicit item factors. Also adds user and item biases: r̂ᵢⱼ = μ + bᵢ + bⱼ + p̃ᵢ·qⱼᵀ. Captures the idea that *what a user has rated* reveals information beyond just the ratings themselves.

**Q8. What is BPR (Bayesian Personalized Ranking)?**  
Optimization criterion for implicit feedback: instead of pointwise regression, optimize *pairwise ranking*. For each user i, item j (interacted) and item k (not interacted): maximize P(r̂ᵢⱼ > r̂ᵢₖ). Loss: -Σ ln σ(r̂ᵢⱼ - r̂ᵢₖ) + regularization. More aligned with the actual evaluation metric (ranking) than MSE on implicit binary data.

**Q9. What is implicit vs explicit feedback? Which is more common in production?**  
**Explicit:** Direct ratings (stars, thumbs up/down). High quality but sparse — most users don't rate.  
**Implicit:** Behavioral signals (clicks, views, purchases, time-spent, add-to-cart). Much more abundant but noisy (a click doesn't mean like; viewing might mean dislike). **Implicit is far more common in production.** Weighting schemes matter: frequency of interaction, recency, action type (purchase >> click).


## 3. Deep Learning for RecSys

**Q10. What is a Two-Tower (Dual-Encoder) model?**  
Two separate deep neural networks: one encodes users, one encodes items, into a shared embedding space. Similarity = dot product (or cosine) of user and item embeddings. **Key advantage:** Item tower runs offline → index all item embeddings in an ANN index (FAISS, ScaNN). At serving: embed query user → ANN search over item index. Scales to millions of items. Used by YouTube, Pinterest, Google Play. Limitations: no cross-feature interaction between user and item features.

**Q11. What is the YouTube DNN recommendation architecture?**  
Two-stage architecture: (1) **Candidate generation:** Two-tower model retrieves top-hundreds of videos from millions. (2) **Ranking:** Deep NN with rich features scores candidates precisely. Candidate generation uses watch history as implicit signal, outputs user embedding via average pooling of video embeddings. Key insight: treat recommendation as extreme multiclass classification (predict next video from all videos).

**Q12. What are Wide & Deep models?**  
Google (2016): Combines (1) **Wide component:** Linear model (logistic regression) — memorization of specific feature combinations (manually crafted cross features). (2) **Deep component:** DNN on embeddings — generalization to unseen combinations. Joint training. Wide handles "users who liked X AND Y always want Z"; Deep generalizes to new patterns. Used in Google Play app recommendations.

**Q13. What is DeepFM?**  
Replaces the "wide" (manual feature crosses) part of Wide & Deep with a Factorization Machine (FM). FM automatically learns all pairwise feature interactions via latent factor dot products without feature engineering. DeepFM = FM (all 2nd-order interactions) + DNN (higher-order interactions). No feature engineering needed; shares embedding layer between FM and DNN.

**Q14. What is the role of attention in recommendation systems?**  
**DIN (Deep Interest Network):** Uses attention to weight historical interactions by relevance to the candidate item — not all past behavior is equally relevant. Attention weight = similarity between historical item and candidate item. **DIN insight:** User interests are diverse and target-dependent; a user who bought books and electronics cares differently for book vs electronics candidates.

**Q15. What are session-based recommendation systems?**  
Recommends based only on the *current session* (no user history across sessions). Important for: anonymous users, contexts where past behavior is irrelevant (mood, context changes). Methods: GRU4Rec (RNN on session), SASRec (self-attention on session), BERT4Rec (bidirectional transformer). Framed as next-item prediction.


## 4. Retrieval, Ranking & Production Systems

**Q16. Explain the multi-stage funnel in production RecSys.**  
```
All Items (millions)
    ↓ Retrieval/Candidate Generation (fast, recall-focused)
Top ~1000 candidates
    ↓ Scoring/Pre-ranking (lightweight model)
Top ~200 candidates  
    ↓ Ranking (complex model, precision-focused)
Top ~50 ranked items
    ↓ Re-ranking (business rules, diversity, freshness)
Final recommendations shown to user
```
Each stage trades off precision for speed. Retrieval uses ANN; ranking uses full feature interaction models.

**Q17. What is ANN (Approximate Nearest Neighbor) search and why is it used?**  
Exact nearest neighbor search is O(N·d) — too slow for millions of items. ANN finds approximately closest items much faster. Methods: **FAISS** (Facebook): IVF (inverted file index) + HNSW (hierarchical navigable small world graph); **ScaNN** (Google): asymmetric quantization. Trade recall for speed. In production, 99% recall at 10× speedup is typically acceptable.

**Q18. What is the Explore-Exploit tradeoff in recommendations?**  
**Exploit:** Recommend items the system is confident the user will like (based on history). **Explore:** Recommend items with uncertainty to gather feedback (improve future recommendations). Pure exploitation leads to filter bubbles; pure exploration gives bad UX.  
Solutions: **ε-greedy** (explore with probability ε), **Thompson Sampling** (sample from posterior), **UCB** (Upper Confidence Bound), **Boltzmann exploration**. Explore/exploit is a multi-armed bandit problem.

**Q19. What is position bias in RecSys and how do you correct for it?**  
Users are more likely to click items shown in top positions (not because they prefer them, but because they see them first). Models trained on click data learn this bias. Correction methods: (1) **Inverse Propensity Scoring (IPS):** Downweight clicks at popular positions. (2) **Unbiased learning-to-rank:** Model position as a separate confound. (3) **Randomization experiments:** Show items in random positions for some users to collect unbiased data. Critical for unbiased recommendation quality.

**Q20. What is feature engineering for RecSys ranking models?**  
Key feature categories:  
- **User features:** Demographics, long-term interests, user embeddings, account age  
- **Item features:** Category, popularity, freshness, price, content embeddings  
- **User-item interaction features:** Past interactions with this item/category, time since last interaction  
- **Context features:** Device, time of day, location, session context  
- **Cross features:** User embedding × item embedding dot product, user-category affinity  
**Feature hashing** and **embedding lookups** are standard for high-cardinality categoricals.


## 5. Evaluation Metrics

**Q21. What metrics are used to evaluate recommendation systems?**  
**Offline metrics:**  
- **Precision@K:** Fraction of top-K recommendations that are relevant  
- **Recall@K:** Fraction of relevant items found in top-K  
- **NDCG@K:** Normalized Discounted Cumulative Gain — accounts for *position* of relevant items (higher rank = more credit): DCG = Σ relᵢ/log₂(i+1)  
- **MRR (Mean Reciprocal Rank):** Average of 1/rank of first relevant item  
- **Hit Rate@K:** Binary — did any relevant item appear in top-K?  
- **AUC:** For ranking (does the model rank positives above negatives?)

**Q22. Why are offline metrics insufficient for RecSys evaluation?**  
Offline metrics measure only on *observed* interactions — but the system only showed certain items to users. Unshown items are not "irrelevant," just unobserved (missing-not-at-random bias). Also: offline metrics don't capture diversity, novelty, serendipity, or long-term user satisfaction. **A/B testing** on live traffic is the gold standard for RecSys evaluation — measuring engagement, retention, revenue.

**Q23. What is NDCG and why is it preferred for ranking evaluation?**  
NDCG@K rewards relevant items ranked higher: DCG@K = Σᵢ₌₁ᴷ relᵢ/log₂(i+1). Normalized by ideal DCG (IDCG — perfect ranking). Range [0,1]. Advantages: (1) position-aware; (2) handles graded relevance (not just binary); (3) normalized across queries. Most common metric for learning-to-rank and RecSys offline evaluation.


## 6. Diversity, Fairness & Special Topics

**Q24. What is the filter bubble problem and how is it addressed?**  
Pure relevance optimization creates filter bubbles: users see more of what they already like → reinforces existing preferences → reduces diversity of exposure. Solutions: (1) **Diversity re-ranking:** MMR (Maximal Marginal Relevance) — balance relevance with dissimilarity to already-selected items. (2) **Intra-list diversity constraints.** (3) **Exploration policies.** (4) **Serendipity rewards** in the objective.

**Q25. What is knowledge graph-based recommendation?**  
Incorporates structured knowledge (item attributes, entity relationships) as a graph. Methods: **RippleNet** (propagate user preferences over KG), **KGCN** (graph conv on KG), **KGAT** (attention-based KG). Addresses cold-start (items with no interactions have KG features), improves interpretability ("recommended because you like author X and this book is also by X").

**Q26. Explain multi-task learning in RecSys.**  
Jointly optimize multiple objectives (CTR, dwell time, purchase, rating). Shared lower layers learn common representations; task-specific heads learn per-objective. Benefits: better generalization, prevents individual objectives overfitting, single model for multiple tasks. Challenge: task conflict — optimizing one may hurt another. Solutions: gradient surgery, uncertainty weighting (Kendall et al.), MMoE (mixture of experts gating per task).

**Q27. What is a feature store in ML/RecSys systems?**  
Centralized repository that stores and serves ML features with low latency. Separates feature computation (batch pipeline) from feature serving (online). Key properties: (1) **point-in-time correctness** — no future leakage for training; (2) **consistency** — same features in training and serving; (3) **low-latency serving** (< 10ms). Examples: Feast, Tecton, Hopsworks. Solves the training-serving skew problem.


## 7. ⚠️ Trick Questions

**Q28. ⚠️ Is a high CTR (click-through rate) always a good optimization target?**  
No — optimizing CTR alone leads to clickbait. A user clicks but doesn't buy, doesn't watch long, or rates the content poorly. Production systems use multi-objective optimization: CTR × watch_time, or CTR + satisfaction signals. YouTube found that optimizing watch time instead of clicks significantly improved user satisfaction. Always consider what you're actually trying to maximize for the business.

**Q29. ⚠️ Can collaborative filtering work with no user history at all?**  
Not directly — CF requires at least some interactions. For users with zero history, fall back to: popularity-based recommendations, content-based on signup information, or ask explicit preferences during onboarding. This is the user cold-start problem. Some approaches use user demographics or device metadata as a proxy.

**Q30. ⚠️ Is matrix factorization the same as SVD?**  
Related but not the same. Exact SVD on the *full* (imputed) matrix is one approach, but it doesn't handle missing data well and is expensive. MF as used in RecSys optimizes only over *observed* entries with regularization, typically via SGD or ALS. It finds a *low-rank approximation* that minimizes prediction error on observed ratings — not the globally optimal matrix decomposition.

**Q31. ⚠️ Does more user data always lead to better recommendations?**  
Not necessarily. Data quality matters more than quantity. Old interactions may be outdated (user preferences change). Privacy-constrained data may introduce selection bias. For session-based systems, too much history can hurt (noise from irrelevant past sessions). Recency weighting and time-aware models address this.

**Q32. ⚠️ Is a two-tower model always better than matrix factorization?**  
Not always. For pure collaborative filtering with sufficient data, MF (especially iALS) is highly competitive with deep two-tower models and much faster to train. Two-tower wins when: rich content features exist, cold-start is critical, or complex non-linear interactions need to be captured. MF wins when: only implicit feedback exists, fast training iteration needed, or serving constraints are tight.

**Q33. ⚠️ What's the difference between retrieval and ranking in a RecSys funnel?**  
**Retrieval** prioritizes *recall* — don't miss relevant items — and must handle millions of items very fast (milliseconds). Uses simple models (dot product similarity, ANN). **Ranking** prioritizes *precision* — put the best items first — on the ~hundreds of candidates from retrieval. Can use expensive feature-rich models. Confusing the two leads to either too slow retrieval or too weak ranking.


## 8. Advanced Topics

**Q34. What is reinforcement learning for recommendation systems?**  
Model the recommendation process as an MDP: state = user context/history, action = recommended item, reward = user feedback (click, watch, purchase). Policy (recommendation strategy) is learned to maximize cumulative long-term reward. Methods: DQN, Actor-Critic, REINFORCE. Challenge: sparse rewards, non-stationary user preferences, off-policy learning from logged data. Used for long-term engagement optimization (avoiding short-term CTR that harms long-term retention).

**Q35. What is federated learning for recommendations?**  
Train recommendation models without centralizing user data. Each device trains on local data, sends only model updates (gradients) to server. Server aggregates (FedAvg). Preserves user privacy, enables GDPR/CCPA compliance. Challenges: communication overhead, heterogeneous data distributions across devices, model convergence. Google uses FL for Gboard keyboard predictions.

**Q36. Explain the concept of embedding everything in RecSys.**  
Modern RecSys maps all entities (users, items, queries, contexts) to dense vectors in a shared embedding space. Item2Vec (Word2Vec applied to item sequences), BERT4Rec (transformer for item sequences), graph embeddings (for social network/KG context). The key insight: entities that co-occur or co-interact should have similar embeddings — enables transfer across tasks and cold-start handling.

**Q37. What is a listwise learning-to-rank loss?**  
**Pointwise:** Treat each item independently (regression or classification). Ignores inter-item relationships.  
**Pairwise:** BPR, LambdaRank — optimize relative ordering of pairs. More aligned with ranking.  
**Listwise:** Optimize the full ranked list directly. LambdaMART: gradient boosted trees with LambdaRank gradients approximating NDCG. SoftMax loss over scores is also listwise. Best alignment with ranking metrics like NDCG but more complex to optimize.


## 9. Quick Reference Summary

### RecSys Algorithm Taxonomy
```
Recommendation Systems
├── Collaborative Filtering
│   ├── Memory-based (User-CF, Item-CF)
│   └── Model-based (MF, SVD++, BPR, NMF)
├── Content-Based Filtering
├── Hybrid (Wide&Deep, DeepFM, Two-Tower)
├── Session-Based (GRU4Rec, SASRec, BERT4Rec)
├── Knowledge-Based
└── Reinforcement Learning-Based
```

### Production Stack
| Layer | Tools/Methods |
|---|---|
| Data pipeline | Spark, Kafka, Flink |
| Feature store | Feast, Tecton, Redis |
| Model training | TensorFlow/PyTorch + Horovod |
| ANN index | FAISS, ScaNN, HNSW |
| Serving | TF Serving, TorchServe, custom gRPC |
| A/B testing | Experimentation platform |

### When to Use What
| Scenario | Approach |
|---|---|
| No content features, enough data | Matrix Factorization (iALS) |
| Rich item/user features | Two-Tower (Dual Encoder) |
| Need feature interactions | DeepFM / Wide & Deep |
| Sequential/session data | SASRec / BERT4Rec |
| Cold-start critical | Content-Based + Hybrid |
| Long-term engagement | RL-based (with caution) |

### Key Metrics Summary
| Metric | Measures | Sensitive to Position? |
|---|---|---|
| Precision@K | Relevance of top-K | No |
| Recall@K | Coverage of relevant items | No |
| NDCG@K | Ranked relevance quality | Yes |
| MRR | First relevant item rank | Yes (only first) |
| Hit Rate@K | Any relevant in top-K | No |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

# Demonstrate User-CF and Item-CF on a toy rating matrix
# Rows=users, Cols=items (0=unrated)
R = np.array([
    [5, 3, 0, 1, 0],
    [4, 0, 4, 1, 2],
    [1, 1, 0, 5, 4],
    [0, 0, 5, 4, 0],
    [0, 1, 5, 4, 0],
], dtype=float)

users = ['Alice', 'Bob', 'Carol', 'Dave', 'Eve']
items = ['Item1', 'Item2', 'Item3', 'Item4', 'Item5']

# User-based CF
user_sim = cosine_similarity(R)
np.fill_diagonal(user_sim, 0)

# Item-based CF  
item_sim = cosine_similarity(R.T)
np.fill_diagonal(item_sim, 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
im1 = ax1.imshow(user_sim, cmap='coolwarm', vmin=0, vmax=1)
ax1.set_xticks(range(5)); ax1.set_yticks(range(5))
ax1.set_xticklabels(users, rotation=45); ax1.set_yticklabels(users)
ax1.set_title('User Similarity Matrix (User-CF)'); plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(item_sim, cmap='coolwarm', vmin=0, vmax=1)
ax2.set_xticks(range(5)); ax2.set_yticks(range(5))
ax2.set_xticklabels(items, rotation=45); ax2.set_yticklabels(items)
ax2.set_title('Item Similarity Matrix (Item-CF)'); plt.colorbar(im2, ax=ax2)

plt.suptitle('Collaborative Filtering: User vs Item Similarity', fontsize=13)
plt.tight_layout()
plt.savefig('cf_similarity.png', dpi=100, bbox_inches='tight')
plt.show()

# Predict Alice's rating for Item3 using User-CF (top-2 neighbors)
target_user, target_item = 0, 2
top_k = 2
neighbors = np.argsort(user_sim[target_user])[::-1][:top_k]
rated = [n for n in neighbors if R[n, target_item] > 0]
if rated:
    weights = user_sim[target_user][rated]
    pred = np.dot(weights, R[rated, target_item]) / weights.sum()
    print(f"User-CF prediction for {users[target_user]}, {items[target_item]}: {pred:.2f}")


In [ ]:
# Matrix Factorization with SGD (from scratch)
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

R = np.array([
    [5, 3, 0, 1, 0],
    [4, 0, 4, 1, 2],
    [1, 1, 0, 5, 4],
    [0, 0, 5, 4, 0],
    [0, 1, 5, 4, 0],
], dtype=float)

n_users, n_items = R.shape
k = 3  # latent factors
lr = 0.01
reg = 0.1
n_epochs = 500

P = np.random.normal(0, 0.1, (n_users, k))  # user factors
Q = np.random.normal(0, 0.1, (n_items, k))  # item factors

observed = [(i, j) for i in range(n_users) for j in range(n_items) if R[i,j] > 0]
losses = []

for epoch in range(n_epochs):
    np.random.shuffle(observed)
    total_loss = 0
    for i, j in observed:
        pred = P[i] @ Q[j]
        err = R[i, j] - pred
        P[i] += lr * (err * Q[j] - reg * P[i])
        Q[j] += lr * (err * P[i] - reg * Q[j])
        total_loss += err**2 + reg * (np.sum(P[i]**2) + np.sum(Q[j]**2))
    losses.append(total_loss / len(observed))

R_pred = P @ Q.T

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
ax1.plot(losses); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('MF Training Loss'); ax1.grid(True, alpha=0.3)

users = ['Alice','Bob','Carol','Dave','Eve']
items = ['Item1','Item2','Item3','Item4','Item5']
ax2.imshow(R, cmap='YlOrRd', vmin=0, vmax=5)
ax2.set_title('Original Matrix (0=missing)')
ax2.set_xticks(range(5)); ax2.set_yticks(range(5))
ax2.set_xticklabels(items, rotation=45, fontsize=8); ax2.set_yticklabels(users, fontsize=8)
for i in range(5):
    for j in range(5):
        ax2.text(j, i, f'{R[i,j]:.0f}', ha='center', va='center', fontsize=9)

im = ax3.imshow(R_pred, cmap='YlOrRd', vmin=0, vmax=5)
ax3.set_title('Predicted Matrix (filled in!)')
ax3.set_xticks(range(5)); ax3.set_yticks(range(5))
ax3.set_xticklabels(items, rotation=45, fontsize=8); ax3.set_yticklabels(users, fontsize=8)
for i in range(5):
    for j in range(5):
        ax3.text(j, i, f'{R_pred[i,j]:.1f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax3)

plt.suptitle('Matrix Factorization with SGD', fontsize=13)
plt.tight_layout()
plt.savefig('matrix_factorization.png', dpi=100, bbox_inches='tight')
plt.show()

# RMSE on observed entries
observed_rmse = np.sqrt(np.mean([(R[i,j] - R_pred[i,j])**2 for i,j in observed]))
print(f"RMSE on observed entries: {observed_rmse:.4f}")


In [ ]:
# NDCG@K implementation and comparison
import numpy as np

def dcg_at_k(relevances, k):
    """Discounted Cumulative Gain at K."""
    relevances = np.array(relevances[:k], dtype=float)
    if len(relevances) == 0:
        return 0.0
    positions = np.arange(1, len(relevances) + 1)
    return np.sum(relevances / np.log2(positions + 1))

def ndcg_at_k(relevances, k):
    """Normalized DCG at K."""
    ideal = sorted(relevances, reverse=True)
    idcg = dcg_at_k(ideal, k)
    if idcg == 0:
        return 0.0
    return dcg_at_k(relevances, k) / idcg

# Example: 3 different ranking systems for same query
# Relevance scores: 3=very relevant, 2=relevant, 1=somewhat, 0=not relevant
true_relevances = [3, 2, 3, 0, 1, 2]  # ground truth

# System A: perfect ranking
perfect = sorted(true_relevances, reverse=True)

# System B: good but not perfect
good = [3, 3, 2, 1, 2, 0]

# System C: poor ranking (relevant items at bottom)
poor = [0, 1, 0, 2, 3, 3]

print("="*50)
print(f"{'System':<15} {'DCG@3':>8} {'NDCG@3':>8} {'DCG@6':>8} {'NDCG@6':>8}")
print("="*50)
for name, ranking in [('Perfect', perfect), ('Good', good), ('Poor', poor)]:
    print(f"{name:<15} {dcg_at_k(ranking,3):>8.3f} {ndcg_at_k(ranking,3):>8.3f} "
          f"{dcg_at_k(ranking,6):>8.3f} {ndcg_at_k(ranking,6):>8.3f}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, ranking) in zip(axes, [('Perfect', perfect), ('Good', good), ('Poor', poor)]):
    colors = ['green' if r >= 2 else 'orange' if r == 1 else 'red' for r in ranking]
    bars = ax.bar(range(1, len(ranking)+1), ranking, color=colors, edgecolor='black', alpha=0.8)
    ax.set_title(f'{name}\nNDCG@3={ndcg_at_k(ranking,3):.3f}, NDCG@6={ndcg_at_k(ranking,6):.3f}')
    ax.set_xlabel('Rank position'); ax.set_ylabel('Relevance')
    ax.set_xticks(range(1, len(ranking)+1)); ax.grid(True, alpha=0.3, axis='y')
    ax.axvline(3.5, color='blue', ls='--', alpha=0.5, label='K=3 cutoff')
    ax.legend(fontsize=8)
plt.suptitle('NDCG@K: Position matters — relevant items ranked higher = better score', fontsize=11)
plt.tight_layout()
plt.savefig('ndcg_comparison.png', dpi=100, bbox_inches='tight')
plt.show()


## 10. Additional Commonly Asked Questions

**Q_A1. What is the difference between explicit and implicit feedback in training objectives?**  
**Explicit feedback training:** Minimize RMSE/MAE on ratings. Direct signal but sparse.  
**Implicit feedback training:** Treat observed interactions as positive signals; unobserved as negative (but unlabeled!). Weighted matrix factorization: high confidence weight on observed (rᵢⱼ=1, wᵢⱼ=1+α·fᵢⱼ where fᵢⱼ=interaction frequency), low weight on unobserved (rᵢⱼ=0, wᵢⱼ=1). BPR: pairwise — observed items should rank above unobserved. Negative sampling: treat random unobserved items as negatives.

**Q_A2. What is embedding-based retrieval and how does it relate to Two-Tower models?**  
Both user and items are mapped to the same embedding space. At serving, all item embeddings are pre-computed and indexed in an ANN structure. Given a user, embed them and run ANN search. The Two-Tower model *learns* these embeddings. Key design choices: (1) embedding dimension (higher = more expressive, slower); (2) similarity function (dot product = scales with norm; cosine = direction only); (3) negative sampling strategy (random vs hard negatives) critically affects quality.

**Q_A3. What is hard negative mining and why is it important?**  
In contrastive learning for two-tower models, random negatives (random items) are "too easy" — the model easily distinguishes them from positives. Hard negatives are items that are similar to the query but *not* the ground truth (e.g., different product from same category). Training with hard negatives forces the model to learn fine-grained distinctions. Methods: in-batch negatives (other items in the same batch), offline mined negatives (items that score high but are not interacted with). Critical for retrieval quality.

**Q_A4. What is the role of popularity bias in recommendation systems?**  
Popular items get more interactions → more training signal → better representations → recommended more → even more interactions (self-reinforcing feedback loop). Results in: (1) long-tail items are underrepresented; (2) recommendations dominated by popular items; (3) reduced diversity and discovery. Mitigations: inverse popularity weighting in loss, separate popularity signal as a feature (not baked into embeddings), debiasing techniques, exploration policies.

**Q_A5. What is click-through rate (CTR) prediction and how does it differ from rating prediction?**  
**CTR prediction:** Binary classification (click vs no-click). Input: user, item, context features. Output: P(click). Very imbalanced (click rates 1-5%). Uses log loss (binary cross-entropy). Focus on real-time, feature-rich models (GBDT, DeepFM, DCN).  
**Rating prediction:** Regression on explicit ratings (1-5 stars). Less imbalanced. More direct quality signal. Uses RMSE/MAE. Both are used in production — CTR for immediate engagement, ratings for preference learning.

**Q_A6. What is DCN (Deep & Cross Network)?**  
Google (2017): Replaces the "wide" (manual cross features) in Wide & Deep with a **cross network** that automatically learns bounded-degree feature interactions. Cross layer: x_{l+1} = x_0 · xₗᵀ · wₗ + bₗ + xₗ. The l-th cross layer captures all interactions up to degree l+1. More systematic than manual feature engineering, more interpretable than deep networks. DCN-V2 (2021) uses a matrix instead of a vector for the cross term, greatly increasing expressiveness.

**Q_A7. What is the SASRec model?**  
Self-Attentive Sequential Recommendation (Kang & McAuley 2018): Applies unidirectional (causal) self-attention to a user's item interaction sequence. Each item attends to all previous items. Models long-range dependencies better than GRU4Rec (RNN). Trained with next-item prediction (similar to causal LM pre-training). Efficient: O(L²·d) but L is usually short (e.g., 50 items in history). State-of-the-art for sequential recommendation.

**Q_A8. ⚠️ TRICK: If a user has watched 1000 videos, which ones matter most for recommendation?**  
Not all interactions equally. Key factors: (1) **Recency:** Recent interactions signal current intent (e.g., last 10 videos > first 990); (2) **Interaction type:** Purchase >> add-to-cart >> click >> view; (3) **Completion rate:** 90% video completion >> 5%; (4) **Explicit feedback:** Thumbs up >> no signal. DIN (Deep Interest Network) uses attention over history weighted by relevance to the candidate item — the system should focus on past watches *similar* to the current candidate. Naive averaging of all 1000 embeddings dilutes the signal.

**Q_A9. ⚠️ TRICK: Is higher Precision@K always better than higher Recall@K?**  
Neither is universally better — they serve different goals. Precision@K: "Are the items I show high quality?" (UX quality). Recall@K: "Am I covering what the user wants?" (coverage). For platforms with large catalogs (Netflix, Spotify), recall is often more important — if you miss the user's actual preference, they churn. For advertising, precision matters more (don't waste impressions). The F1@K or NDCG@K balance both.

**Q_A10. How do you handle the feedback loop / data distribution shift in recommendation systems?**  
RecSys creates its own training data: model recommends items → users interact (or not) → data is collected → model is retrained. But data is *missing-not-at-random* — we only observe interactions on *shown* items. Solutions: (1) **Inverse propensity scoring (IPS):** Downweight frequently shown items; (2) **Counterfactual evaluation:** Estimate what would have happened with a different policy; (3) **Exploration:** Occasionally show random/non-greedy items to gather unbiased data; (4) **Continuous retraining with recency weighting** to adapt to distribution shifts.
